<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2025/2026  

<img src="../../img/portada.png" width="300"/>

---

#  Aprendizaje automático relacional  
### Primera Convocatoria – Junio 2026  

**Profesor:** Pedro Almagro Blanco

**Autores:**  
David Espina Apellaniz  
Marco Padilla Gómez  

**Fecha de entrega:** 8 de Junio de 2025  

</div>

# 04 - Random Forest: experimentos de hiperparámetros

En este notebook se prueban diferentes configuraciones de **Random Forest**.

Se comparan valores de `n_estimators` y `max_depth` para seleccionar la combinación con mejor rendimiento según `f1`.

## Objetivo metodologico

Este cuaderno ajusta hiperparametros de Random Forest. Se comparan distintos numeros de arboles y profundidades maximas para equilibrar rendimiento y complejidad.


## 1. Importación de librerías y configuración inicial

Se importan las funciones de carga, división de datos, evaluación y guardado de modelos.

In [1]:
from pathlib import Path
import sys

import pandas as pd

sys.path.append("../../")

from src.data_loader import load_processed_dataset
from src.evaluation import evaluate_model, save_metrics, save_model
from src.model_training import get_base_models, split_data

ROOT = Path("../../").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Entrenamiento con distintas configuraciones

Se prueban diferentes números de árboles y profundidades máximas para analizar su impacto en el rendimiento.

### Hiperparametros evaluados

`n_estimators` controla cuantos arboles forman el bosque. `max_depth` limita la profundidad de cada arbol. Un bosque muy profundo puede mejorar el entrenamiento, pero no siempre mejora la generalizacion.


In [2]:
df = load_processed_dataset(DATA_PROCESSED / "twitch_mature_features.csv")
X = df.drop(columns=["new_id", "mature"])
y = df["mature"]
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
base = get_base_models(random_state=42)["random_forest"]
experiments = []
for n_estimators in [100, 200]:
    for max_depth in [None, 10, 20]:
        candidate = base.__class__(
            random_state=42,
            class_weight="balanced",
            n_estimators=n_estimators,
            max_depth=max_depth,
        )
        candidate.fit(X_train, y_train)
        experiments.append({
            "model": "random_forest",
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            **evaluate_model(y_test, candidate.predict(X_test))
        })
results_df = pd.DataFrame(experiments)
best_row = results_df.sort_values("f1", ascending=False).iloc[0]
best_max_depth = None if pd.isna(best_row["max_depth"]) else int(best_row["max_depth"])

best_model = base.__class__(
    random_state=42,
    class_weight="balanced",
    n_estimators=int(best_row["n_estimators"]),
    max_depth=best_max_depth,
)
best_model.fit(X_train, y_train)
results_df.sort_values("f1", ascending=False)

,model,n_estimators,max_depth,accuracy,precision,recall,f1
4,random_forest,200,10.0,0.702151,0.490040,0.452206,0.470363
1,random_forest,100,10.0,0.701075,0.488000,0.448529,0.467433
3,random_forest,200,NaN,0.723656,0.577320,0.205882,0.303523
0,random_forest,100,NaN,0.721505,0.565657,0.205882,0.301887
2,random_forest,100,20.0,0.715054,0.533333,0.205882,0.297082
5,random_forest,200,20.0,0.717204,0.544554,0.202206,0.294906


## 3. Guardado de resultados y mejor modelo

Se guardan los resultados de los experimentos y el mejor modelo Random Forest.

In [3]:
save_metrics(results_df.to_dict(orient="records"), RESULTS_DIR / "rf_grid_metrics.csv")
save_model(best_model, ROOT / "models" / "random_forest_grid.joblib")

## 4. Conclusiones

El ajuste de hiperparámetros permite seleccionar un Random Forest que consiga buen rendimiento sin aumentar innecesariamente la complejidad del modelo.